In [ ]:
import os
from huggingface_hub import login

HF_TOKEN = "replace with your token"  # Replace with your actual token
os.environ['HF_TOKEN'] = HF_TOKEN
login(token=HF_TOKEN)

c:\Users\lenovo\anaconda3\envs\nik\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [ ]:
!pip install datasets
!pip install bitsandbytes

In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"

In [ ]:
import os
os.environ["BITSANDBYTES_NOWELCOME"] = "1"  # Disable CUDA welcome message and force CPU offloading
import torch
from transformers import AutoTokenizer, Gemma3ForCausalLM, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from datasets import Dataset
import pandas as pd

# ... (rest of your code)


# 1. Prepare Your Dataset
# Assuming your data is in a Pandas DataFrame with a 'text' column
data = {'text': []}
df = pd.DataFrame(data)
raw_dataset = Dataset.from_pandas(df)

# If your data is in a text file (one example per line):
# raw_dataset = Dataset.from_text("your_data.txt")

# If your data is in a CSV file with a 'text' column:
# raw_dataset = load_dataset("csv", data_files="your_data.csv")

# If your data is in a JSON file with a 'text' field:
# raw_dataset = load_dataset("json", data_files="your_data.json")

# 2. Load the Tokenizer
ckpt = "google/gemma-3-1b-it-qat-q4_0-unquantized"
tokenizer = AutoTokenizer.from_pretrained(ckpt)

# 3. Tokenize Your Dataset
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=512,  # <-- Add this line
        padding="max_length"  # Optional: enables uniform sequence lengths
    )


tokenized_datasets = raw_dataset.map(tokenize_function, batched=True)

# 4. Prepare for Training
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir="./gemma-finetuned",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-5,
    num_train_epochs=1,
    bf16=True,
    optim="adafactor",    # <--- new
    logging_dir="./logs",
    logging_steps=10,
    save_steps=500,
    save_total_limit=2,
    push_to_hub=False,
    report_to=None,
)



# Load the model *after* defining training arguments to ensure it's used in the Trainer
model = Gemma3ForCausalLM.from_pretrained(
    ckpt,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets,
    data_collator=data_collator,
)

# 5. Start Training
trainer.train()

# 6. Save Your Fine-tuned Model
trainer.save_model("/kaggle/working/gemma-finetuned-final")
tokenizer.save_pretrained("/kaggle/working/gemma-finetuned-final")

print("Fine-tuning complete! Model and tokenizer saved to ./gemma-finetuned-final")